# Part B — A long wave over variable bathymetry

**Working time:** about 3 hours  
**Work in groups of 2–4.** This notebook is guided and is not submitted.

Part B investigates a tsunami-like long disturbance. It is not a hazard
or inundation model. Every grid cell remains wet, including the final
coastal cell.

## Learning goals

- Relate local long-wave speed to local water depth.
- Use virtual gauges to compare arrival time and surface elevation.
- Conduct one controlled bathymetry experiment.
- Learn how to change wind forcing, bathymetry, domain size, and initial state.
- Prepare a focused proposal for Part C.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from shallowwater import (
    ModelParams, backend_info, compute_dt_cfl, depth_on_u,
    load_bathymetry, make_grid,
    make_wind_forcing_from_file, run_model, shelf_bathymetry,
    uniform_wind_forcing, zero_forcing,
)

print(backend_info())
DATA_DIR = next(
    path for path in (Path("../data"), Path("data"), Path("MT1562_python_lab_waves/data"))
    if path.exists()
)
print("Course data:", DATA_DIR.resolve())


## 1. Build a shelf and make a prediction

The basin is deep in the west and shallow near the eastern wall. The
disturbance is broad and nearly one-dimensional, which reduces geometric
spreading and makes the depth effect easier to isolate.


In [ ]:
Nx, Ny = 160, 24
Lx, Ly = 2.4e6, 360e3
grid = make_grid(Nx, Ny, Lx, Ly)
H = shelf_bathymetry(
    grid, H_deep=3000.0, H_coast=120.0,
    shelf_width=800e3, coast="east", power=1.5,
)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(grid.x_c / 1e3, H[Ny // 2], linewidth=2)
ax.invert_yaxis()
ax.set(xlabel="x [km]", ylabel="water depth [m]", title="Model bathymetry")
ax.grid(alpha=0.25)
plt.show()

c_deep = np.sqrt(9.81 * 3000.0)
c_coast = np.sqrt(9.81 * 120.0)
print(f"deep-water long-wave speed: {c_deep:.1f} m/s")
print(f"coastal long-wave speed:    {c_coast:.1f} m/s")


**Prediction 1.** Where will the wave travel fastest? What changes do you expect as it crosses the shelf?

**Your response:**


In [ ]:
def cross_basin_pulse(grid, params, *, amplitude=0.12, radius=80e3, x0=350e3):
    eta_line = amplitude * np.exp(-((grid.x_c - x0) / radius) ** 2)
    eta = np.repeat(eta_line[None, :], grid.Ny, axis=0)
    H_u = depth_on_u(grid, params.H)
    eta_u = amplitude * np.exp(-((grid.x_u - x0) / radius) ** 2)
    u = np.repeat(eta_u[None, :], grid.Ny, axis=0) * np.sqrt(params.g / H_u)
    v = np.zeros((grid.Ny + 1, grid.Nx))
    return eta, u, v


def run_shelf_case(H_coast, *, tmax_hours=7.0):
    depth = shelf_bathymetry(
        grid, H_deep=3000.0, H_coast=H_coast,
        shelf_width=800e3, coast="east", power=1.5,
    )
    params = ModelParams(H=depth, g=9.81, f0=0.0, beta=0.0, r=0.0, linear=True)
    dt = compute_dt_cfl(grid, params, cfl=0.42)
    out = run_model(
        tmax=tmax_hours * 3600,
        dt=dt,
        grid=grid,
        params=params,
        forcing_fn=zero_forcing,
        ic_fn=lambda g, p: cross_basin_pulse(g, p),
        save_every=5,
        out_vars=("eta",),
    )
    return params, out


params_120, out_120 = run_shelf_case(120.0)
eta_120 = np.asarray(out_120["eta"])
times_120 = np.asarray(out_120["time"])
eta_line_120 = eta_120.mean(axis=1)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
image = ax.pcolormesh(
    grid.x_c / 1e3, times_120 / 3600, eta_line_120,
    shading="auto", cmap="RdBu_r"
)
ax.set(xlabel="x [km]", ylabel="time [hours]", title="Wave crossing the shelf")
fig.colorbar(image, ax=ax, label="surface displacement [m]")
plt.show()


## 2. Virtual gauges

Four gauges sample the deep basin, the shelf break, the slope, and the
coastal region. The largest peak is not always the first arrival, so use
both the Hovmöller diagram and the time series.


In [ ]:
gauge_x_km = [700, 1400, 1900, 2250]
gauge_indices = [int(np.argmin(abs(grid.x_c / 1e3 - x))) for x in gauge_x_km]

fig, ax = plt.subplots(figsize=(9, 4))
for x_km, index in zip(gauge_x_km, gauge_indices):
    ax.plot(times_120 / 3600, eta_line_120[:, index], label=f"{x_km} km")
ax.set(xlabel="time [hours]", ylabel="surface displacement [m]", title="Virtual gauges")
ax.legend(ncol=2)
ax.grid(alpha=0.25)
plt.show()

peak_times = []
for x_km, index in zip(gauge_x_km, gauge_indices):
    signal = eta_line_120[:, index]
    peak_index = int(np.argmax(signal))
    peak_times.append(times_120[peak_index] / 3600)
    print(f"gauge {x_km:4.0f} km: largest positive peak at {peak_times[-1]:.2f} h, "
          f"eta={signal[peak_index]:.3f} m")


**Analysis 1.** Use the plots and gauge records to explain how depth affected propagation. Include at least one number.

**Your response:**


## 3. Controlled coastal-depth experiment

Repeat the case with a 300 m coastal depth. Everything else remains the
same. Compare arrival time and the modeled elevation at the final gauge.


**Prediction 2.** Will the 300 m coastal case arrive earlier or later than the 120 m case? What do you expect for elevation?

**Your response:**


In [ ]:
params_300, out_300 = run_shelf_case(300.0)
eta_line_300 = np.asarray(out_300["eta"]).mean(axis=1)
times_300 = np.asarray(out_300["time"])
coastal_index = gauge_indices[-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_120 / 3600, eta_line_120[:, coastal_index], label="coastal depth 120 m")
ax.plot(times_300 / 3600, eta_line_300[:, coastal_index], label="coastal depth 300 m")
ax.set(xlabel="time [hours]", ylabel="surface displacement [m]",
       title="Near-coast model cell: controlled comparison")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

for label, times, signal in (
    ("120 m", times_120, eta_line_120[:, coastal_index]),
    ("300 m", times_300, eta_line_300[:, coastal_index]),
):
    index = int(np.argmax(signal))
    print(f"{label}: peak time={times[index]/3600:.2f} h, peak eta={signal[index]:.3f} m")


**Analysis 2.** Summarize the controlled comparison. Why must it not be interpreted as a prediction of coastal danger?

**Your response:**


## 4. Bathymetry supplied as a file

The package accepts an in-memory depth array directly. Version 0.1.4 also
provides a validated loader for `.npy`, `.npz`, `.csv`, and `.txt` maps.
Maps must match the grid exactly, use positive depth in metres, and use
array order `(y, x)`. No interpolation or land mask is applied.


In [ ]:
H_file = load_bathymetry(DATA_DIR / "example_bathymetry.npz", grid)
print("shape:", H_file.shape, "minimum depth:", H_file.min(), "maximum depth:", H_file.max())

fig, ax = plt.subplots(figsize=(8, 3.5))
image = ax.imshow(
    H_file, origin="lower", extent=[0, grid.Lx/1e3, 0, grid.Ly/1e3],
    aspect="auto", cmap="Blues"
)
ax.set(xlabel="x [km]", ylabel="y [km]", title="Bathymetry loaded from file")
fig.colorbar(image, ax=ax, label="depth [m]")
plt.show()


**Check.** Describe one difference between the analytic shelf and the file-loaded map.

**Your response:**


## 5. Wind forcing supplied as a file

A forcing file contains static cell-centred wind-stress maps `tau_x` and
`tau_y` in N m(^{-2}). The file is read once and converted to a forcing
function. A separate time envelope controls ramp-up or shut-off.


In [ ]:
wind_from_file = make_wind_forcing_from_file(
    DATA_DIR / "example_wind_forcing.npz",
    grid,
    envelope=lambda t: min(1.0, t / (2 * 3600)),
)
taux, tauy, _ = wind_from_file(2 * 3600, grid, params_120)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
for ax, field, title in zip(
    axes, (taux[:, :-1], tauy[:-1, :]), ("eastward stress", "northward stress")
):
    image = ax.imshow(field, origin="lower", aspect="auto", cmap="RdBu_r")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label="N m$^{-2}$")
plt.show()


### Short wind-forced demonstration

Here the model starts from rest. An eastward wind ramps up, is switched
off after six hours, and pushes water toward the eastern wall.


In [ ]:
wind_params = ModelParams(
    H=H, g=9.81, f0=0.0, beta=0.0,
    r=1/(2*86400), linear=True,
)
wind_dt = compute_dt_cfl(grid, wind_params, cfl=0.42)
wind_forcing = lambda t, g, p: uniform_wind_forcing(
    t, g, p, tau_x=0.08, tau_y=0.0,
    t_ramp=2*3600, t_off=6*3600,
)
wind_out = run_model(
    tmax=8*3600, dt=wind_dt, grid=grid, params=wind_params,
    forcing_fn=wind_forcing,
    ic_fn=lambda g, p: (
        np.zeros((g.Ny, g.Nx)),
        np.zeros((g.Ny, g.Nx+1)),
        np.zeros((g.Ny+1, g.Nx)),
    ),
    save_every=8, out_vars=("eta",),
)
wind_eta = np.asarray(wind_out["eta"])
wind_times = np.asarray(wind_out["time"])
shutoff_index = int(np.argmin(abs(wind_times - 6*3600)))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(
    grid.x_c/1e3, wind_eta[shutoff_index].mean(axis=0),
    label="near wind shut-off (setup)",
)
ax.plot(
    grid.x_c/1e3, wind_eta[-1].mean(axis=0),
    label="two hours later (free response)",
)
ax.axhline(0, color="0.4", linewidth=0.8)
ax.set(xlabel="x [km]", ylabel="surface displacement [m]",
       title="Wind setup and release")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


**Observation 3.** Which coast gains water under eastward wind? What happens after the wind stops?

**Your response:**


## 6. Controls available in Part C

You may investigate one main factor:

| Category | Examples of controls |
|---|---|
| Wind | amplitude, direction, duration, spatial map |
| Bottom | uniform depth, shelf depth/width, ridge, file map, damping |
| Domain | `Lx`, `Ly`, aspect ratio, resolution |
| Initial state | amplitude, radius, position, circular or cross-basin shape |
| Optional | forcing period or rotation |

Changing domain size while keeping `Nx` fixed also changes `dx`; always
report both. Change one primary factor unless the instructor approves a
two-factor experiment.

## 7. Part C proposal


**Project question.** State a question that can be answered with a baseline and two controlled variations.

**Your response:**


**Prediction and mechanism.** What do you expect, and which physical argument supports it?

**Your response:**


**Experimental design.** List the baseline, two variations, the primary parameter, and the diagnostic you will use.

**Your response:**
